## CREATE CHUNKS OF DATA

In [5]:
       
def _to_parquet_safe(df_out: pd.DataFrame, path: str | Path):
    try:
        df_out.to_parquet(path, engine="pyarrow", index=False)
    except Exception:
        try:
            df_out.to_parquet(path, engine="fastparquet", index=False)
        except Exception as e:
            raise RuntimeError(
                "Saving to Parquet failed. Please install 'pyarrow' or 'fastparquet'. "
                f"Original error: {e}"
            )

In [6]:
import pandas as pd
import numpy as np

def build_passages_token_based(
    df,
    context_cols,
    doc_id_col="question_id",
    chunk_size=128,
    overlap_ratio=0.1,
    dedupe=True,
    keep_source=False
):
    """
    Robust passage builder:
    - If df already looks like a passages table (has 'text'), it will use that automatically
      when requested context_cols are missing.
    - Uses tokenizer overflow windows (<=512 tokens) to avoid long-sequence warnings.
    """

    context_cols = list(context_cols)

    # --- If the requested context columns are missing, auto-fallback to 'text' ---
    missing_ctx = [c for c in context_cols if c not in df.columns]
    if missing_ctx:
        if "text" in df.columns:
            # df is already a passages dataframe
            context_cols = ["text"]
        else:
            raise KeyError(
                f"context_cols missing from df.columns: {missing_ctx}. "
                f"Available: {list(df.columns)[:30]}..."
            )

    # --- doc_id handling: fallback to 'doc_id' if question_id isn't present ---
    if doc_id_col not in df.columns and "doc_id" in df.columns:
        doc_id_col = "doc_id"

    has_doc_id = (doc_id_col in df.columns)

    # --- Determine max len (E5 is 512) ---
    MAX_LEN = getattr(tokenizer, "model_max_length", 512)
    if MAX_LEN is None or MAX_LEN > 100000:
        MAX_LEN = 512
    MAX_LEN = min(int(MAX_LEN), 512)

    # Respect caller chunk_size but never exceed model limit
    chunk_size = min(int(chunk_size), MAX_LEN)

    # Stride = overlap in tokens
    stride = int(chunk_size * overlap_ratio)
    if stride < 0:
        stride = 0
    if stride >= chunk_size:
        stride = max(0, chunk_size - 1)

    rows = []

    # Iterate only on columns that exist
    cols = ([doc_id_col] if has_doc_id else []) + context_cols

    for r in df[cols].itertuples(index=True, name=None):
        idx = r[0]

        if has_doc_id:
            doc_id_val = r[1]
            doc_id = doc_id_val if pd.notna(doc_id_val) else idx
            ctx_values = r[2:]
        else:
            doc_id = idx
            ctx_values = r[1:]

        parts = [str(x) for x in ctx_values if pd.notna(x)]
        text = "\n".join(parts).strip()
        if not text:
            continue

        enc = tokenizer(
            text,
            add_special_tokens=False,
            return_offsets_mapping=True,
            truncation=True,
            max_length=chunk_size,
            return_overflowing_tokens=True,
            stride=stride
        )

        offset_windows = enc.get("offset_mapping", [])
        if not offset_windows:
            continue

        # Safety cap: number of chunks per doc
        MAX_CHUNKS_PER_DOC = 20000
        if len(offset_windows) > MAX_CHUNKS_PER_DOC:
            offset_windows = offset_windows[:MAX_CHUNKS_PER_DOC]

        for i, offsets in enumerate(offset_windows):
            if not offsets:
                continue

            char_start = offsets[0][0]
            char_end = offsets[-1][1]
            chunk_text = text[char_start:char_end].strip()
            if not chunk_text:
                continue

            passage_id = f"{doc_id}::r{idx}::c{i}"

            rec = {"passage_id": passage_id, "doc_id": doc_id, "text": chunk_text}
            if keep_source:
                rec["source"] = context_cols[0] if context_cols else None
            rows.append(rec)

    passages = pd.DataFrame(rows)
    if dedupe and not passages.empty:
        passages = passages.drop_duplicates(subset=["text"]).reset_index(drop=True)

    return passages


In [7]:
# --- BUILD & SAVE PASSAGES (token-based chunking) ---
from pathlib import Path
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-small-v2"
model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = 512
tokenizer = model.tokenizer

# Build passages *only from contexts*
passages = build_passages_token_based(
    df=df,
    context_cols=[ctx_col_name],   # e.g. ["context_clean"] also works if that's your column
    doc_id_col="question_id",
    chunk_size=min(CHUNK_SIZE, model.max_seq_length),
    overlap_ratio=OVERLAP_RATIO,
    dedupe=True,
    keep_source=False
)

dups = passages["passage_id"].duplicated().sum()
print("Duplicate passage_id count:", dups)
assert dups == 0, "passage_id must be unique for retrieval mapping"


_to_parquet_safe(passages, PASSAGES_PATH)

n_docs = df.shape[0]
n_pass = len(passages)
avg_chunks = (n_pass / n_docs) if n_docs else 0

print(f"Saved {n_pass:,} passages to '{PASSAGES_PATH}'. "
      f"(chunk_size={CHUNK_SIZE}, overlap_ratio={OVERLAP_RATIO})")
print(f"Docs: {n_docs:,} | Avg chunks/doc: {avg_chunks:.2f}")




Duplicate passage_id count: 0
Saved 10,938 passages to 'C:\Users\Harman\Documents\GISMA\Term_4\Thesis_Redo\Thesis_Implementation\rag_env\Project\rag-laptop\data\processed\passages.parquet'. (chunk_size=512, overlap_ratio=0.25)
Docs: 449 | Avg chunks/doc: 24.36


,passage_id,doc_id,text
0,dpql_3252::r0::c0,dpql_3252,Kent is a county in South East England and one of the home counties It borders Greater London to the north west Surrey to the west East Sussex to the south west and across the Thames Estuary is the county of Essex The county town is Maidstone With a population of over 1 4 million Kent is the second most populous county in the South East region after Hampshire Canterbury Cathedral in Kent has been the seat of the Archbishop of Canterbury leader of the Church of England since the conversion of England to Christianity by Saint Augustine in the 6th century Between London and the Strait of Dover which separates it from mainland Europe Kent has seen both diplomacy and conflict ranging from the Leeds Castle peace talks of 1978 and 2004 to the Battle of Britain in World War II England relied on the county s ports to provide warships through much of its history the Cinque Ports in the 12th 14th centuries and Chatham Dockyard in the 16th 20th centuries were of particular importance France can be seen clearly in fine weather from Folkestone and the White Cliffs of Dover Hills in the form of the North Downs and the Greensand Ridge span the length of the county and in the series of valleys in between and to the south are most of the county s 26 castles Because of its relative abundance of fruit growing and hop gardens Kent is known as The Garden of England The Guardian 1 June 2006 BBC 1 June 2006 The title was defended in 2006 when a survey of beautiful counties by the UKTV Style Gardens channel put Kent in fifth place behind North Yorkshire Devon Derbyshire and Gloucestershire Kent s economy is greatly diversified Haulage logistics and tourism are major industries throughout the county major industries in north west Kent include aggregate building materials printing and scientific research Coal mining has also played its part in Kent s industrial heritage Large parts of Kent are within the London commuter belt and its strong transport connections to the capital and the nearby continent makes Kent a high income county Twenty eight per cent of the county forms part of two Areas of Outstanding Natural Beauty the North Downs and The Weald History The area has been occupied since the Palaeolithic era as attested by finds from the quarries at Swanscombe The Medway megaliths were built during the Neolithic era There is a rich sequence of Bronze Age Iron Age and Roman era occupation as indicated by finds and features such as the Ringlemere gold cup and the Roman villas of the Darent valley The modern name of Kent is derived from the Brythonic word Cantus meaning rim or border This describes the eastern part of the current county area as a border land or coastal district Julius Caesar had described the area as Cantium or home of the
1,dpql_3252::r0::c1,dpql_3252,North Downs and The Weald History The area has been occupied since the Palaeolithic era as attested by finds from the quarries at Swanscombe The Medway megaliths were built during the Neolithic era There is a rich sequence of Bronze Age Iron Age and Roman era occupation as indicated by finds and features such as the Ringlemere gold cup and the Roman villas of the Darent valley The modern name of Kent is derived from the Brythonic word Cantus meaning rim or border This describes the eastern part of the current county area as a border land or coastal district Julius Caesar had described the area as Cantium or home of the Cantiaci in 51 BC The extreme west of the modern county was by the time of Roman Britain occupied by Iron Age tribes known as the Regnenses East Kent became a kingdom of the Jutes during the 5th century and was known as Cantia from about 730 and recorded as Cent in 835 The early medieval inhabitants of the county were known as the Cantwara or Kent people These people regarded the city of Canterbury as their capital In 597 Pope Gregory I appointed the religious missionary as the first Archbishop of Canterbury In the previo